In [8]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import torchvision.models as model

model = model.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 71.3MB/s]


In [9]:
SEED=42
torch.manual_seed(SEED)
np.random.seed(SEED)

Preparing the Image

In [10]:
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.486,0.456,0.406),
        std=(0.229,0.225,0.224)
)])
train_data=datasets.CIFAR10(root='./data',train=True,download=True,transform=transform)
test_data=datasets.CIFAR10(root='./data',train=False,download=True,transform=transform)

train_loader=DataLoader(train_data,batch_size=128,shuffle=True)
test_loader=DataLoader(test_data,batch_size=128)

In [11]:
print(model.features)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [12]:
for param in model.features.parameters():
  param.requires_grad=False

In [13]:
model.classifier[6]=nn.Linear(4096,10)

In [15]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

Evaluation Loop

In [16]:
def eval_model(model,loader,device):
  model.eval()
  correct=0
  total=0
  with torch.no_grad():
    for images,labels in loader:
      images,labels=images.to(device),labels.to(device)
      output=model(images)
      _,predicted=torch.max(output,1)
      total+=labels.size(0)
      correct+=(predicted==labels).sum().item()
    accuracy=100*correct/total
    return accuracy

Traning the model

In [20]:
epochs=1 #change it
for epoch in range(epochs):
  model.train()
  total_loss=0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"For Epoch: {epoch+1}")
  print(f"The total loss is: {total_loss:.3f}")
  print(f"The average loss is: {total_loss/len(train_loader):.3f}")

For Epoch: 1
The total loss is: 236.991
The average loss is: 0.606


In [21]:
acc1=eval_model(model,test_loader,device)
print(f"The accuracy of the model is: {acc1}")

The accuracy of the model is: 85.7


Fine-Tuning model

In [22]:
for param in model.features[-6:].parameters():
  param.requires_grad=True

In [23]:
optimizer=optim.Adam(model.parameters(),lr=1e-5)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=model.to(device)
criterion=nn.CrossEntropyLoss()

In [24]:
epochs=1
for epoch in range(epochs):
  model.train()
  total_loss=0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f"The total loss in epoch: {epoch+1} is {total_loss:.3f}")
  print(f"The total loss per batch is: {total_loss/len(train_loader):.3f}")

The total loss in epoch: 1 is 102.552
The total loss per batch is: 0.262


In [25]:
acc2=eval_model(model,test_loader,device)
print(f"The accuracy of the model is {acc2}")

The accuracy of the model is 88.09
